<h1>Table of Contents<span class="tocSkip"></span></h1>
<div class="toc"><ul class="toc-item"><li><span><a href="#Random-Demands" data-toc-modified-id="Random-Demands-1"><span class="toc-item-num">1&nbsp;&nbsp;</span>Random Demands</a></span></li><li><span><a href="#NN-with-primal-dual--KKT-Style" data-toc-modified-id="NN-with-primal-dual--KKT-Style-2"><span class="toc-item-num">2&nbsp;&nbsp;</span>NN with primal-dual  KKT Style</a></span><ul class="toc-item"><li><span><a href="#Naive-training-(without-warmstarts)" data-toc-modified-id="Naive-training-(without-warmstarts)-2.1"><span class="toc-item-num">2.1&nbsp;&nbsp;</span>Naive training (without warmstarts)</a></span></li></ul></li><li><span><a href="#NN-framework-for-feasibility-only" data-toc-modified-id="NN-framework-for-feasibility-only-3"><span class="toc-item-num">3&nbsp;&nbsp;</span>NN framework for feasibility only</a></span><ul class="toc-item"><li><span><a href="#Naive-Training" data-toc-modified-id="Naive-Training-3.1"><span class="toc-item-num">3.1&nbsp;&nbsp;</span>Naive Training</a></span></li></ul></li><li><span><a href="#NN-with-hard-KKT-layer" data-toc-modified-id="NN-with-hard-KKT-layer-4"><span class="toc-item-num">4&nbsp;&nbsp;</span>NN with hard KKT layer</a></span><ul class="toc-item"><li><span><a href="#Training" data-toc-modified-id="Training-4.1"><span class="toc-item-num">4.1&nbsp;&nbsp;</span>Training</a></span></li><li><span><a href="#Hybrid-NN:-a-verification-oriented-feasibility-network-with-simple-affine/ReLU/clamp-structure-and-hard-slack-enforcement." data-toc-modified-id="Hybrid-NN:-a-verification-oriented-feasibility-network-with-simple-affine/ReLU/clamp-structure-and-hard-slack-enforcement.-4.2"><span class="toc-item-num">4.2&nbsp;&nbsp;</span>Hybrid NN: a verification-oriented feasibility network with simple affine/ReLU/clamp structure and hard slack enforcement.</a></span></li><li><span><a href="#Training" data-toc-modified-id="Training-4.3"><span class="toc-item-num">4.3&nbsp;&nbsp;</span>Training</a></span></li></ul></li><li><span><a href="#Rough" data-toc-modified-id="Rough-5"><span class="toc-item-num">5&nbsp;&nbsp;</span>Rough</a></span></li><li><span><a href="#Rough" data-toc-modified-id="Rough-6"><span class="toc-item-num">6&nbsp;&nbsp;</span>Rough</a></span></li><li><span><a href="#GNN" data-toc-modified-id="GNN-7"><span class="toc-item-num">7&nbsp;&nbsp;</span>GNN</a></span></li></ul></div>

In [1]:
import numpy as np
import pandas as pd
from sys import stderr
from numpy import zeros, arange, isscalar,diag, dot,eye, ix_, ones, r_, pi, flatnonzero as find
from scipy.sparse import csr_matrix as sparse
from numpy.linalg import solve
import pickle
import pyomo.environ as pyo

# Data from the excel case files

In [2]:
# Not sure where this was used in the old code
BUS_TYPE = 1
REF = 3
BUS_I = 0
F_BUS = 1
T_BUS = 2
BR_X = 4
TAP = 9
SHIFT = 10
BR_STATUS = 12

In [18]:
# === Initialization ===
case_name = 'pglib_opf_case3_lmbd'
case_path = f'../excel_outputs/{case_name}.xlsx'
case = pd.read_excel(case_path, sheet_name=['baseMVA','bus','gen','gencost','branch'])

bus_to_idx = {bus: i+1 for i, bus in enumerate(case['bus'].bus_i.values)}
bus_idx = [bus_to_idx[bus] for bus in case['bus'].bus_i.values]
case['bus'].bus_i = case['bus'].bus_i.replace(bus_to_idx) # rename the bus for making PTDF
case['gen'].bus_i = case['gen'].bus_i.replace(bus_to_idx)
baseMVA = case['baseMVA'].values[0][0]

# remove generators and costgen with pmax and pmin equal to zero
zero_gen_idx = []
for num,i in enumerate(case['gen'].Pmax.values/ baseMVA):
    if (i == 0 and (case['gen'].Pmin.values / baseMVA)[num] == 0) or (case['gen'].Pmin.values / baseMVA)[num] < 0:
        zero_gen_idx.append(num)
        
case['gen'].drop(index=zero_gen_idx, inplace=True) # drop


case['gencost'].drop(index=zero_gen_idx, inplace=True) # drop

case['branch'].bus_i = case['branch'].bus_i.replace(bus_to_idx)
case['branch'].bus_j = case['branch'].bus_j.replace(bus_to_idx)

# calculate susceptance, conductance, admittance-square y_sq
case['branch']['susceptance'] = case['branch']['x']/(case['branch']['x']**2+case['branch']['r']**2)
case['branch']['conductance'] = case['branch']['r']/(case['branch']['x']**2+case['branch']['r']**2)
case['branch']["y_sq"] = 1 / (case['branch']['x']**2+case['branch']['r']**2)

nbus = case['bus'].shape[0]
ngen = case['gen'].shape[0]
nbranch = case['branch'].shape[0]


# Data for QCQP

In [19]:
import numpy as np
import pandas as pd

# Branch data
branches = case['branch']
buses = case['bus'].set_index('bus_i')

# Extract from and to buses
fbus = branches['bus_i'].values
tbus = branches['bus_j'].values

# Base voltages (kV)
V_from = buses.loc[fbus, 'baseKV'].values
V_to   = buses.loc[tbus, 'baseKV'].values

# Thermal limit in MVA
S_max = branches['rateA'].values  # normal rating

# Compute currents at both ends (three-phase)
I_from = S_max * 1e6 / (np.sqrt(3) * V_from * 1e3)  # Amps
I_to   = S_max * 1e6 / (np.sqrt(3) * V_to * 1e3)    # Amps

# Absolute maximum current
I_max = np.maximum(I_from, I_to)

# Optional: add it to branch DataFrame
branches['I_max'] = I_max

In [ ]:
def build_G_B_from_branch(n, branch_df, Nb):
    # generates Gn and Bn for a given n
    
    G = np.zeros((Nb, Nb))
    B = np.zeros((Nb, Nb))

    for _, row in branch_df.iterrows():
        i = int(row['bus_i'])
        j = int(row['bus_j'])
        
        if i == n:
#             print(i,j,n,1)
            g = row['susceptance']
            b = row['conductance']
            G[n-1, j-1] = g   # index conversion
            B[n-1, j-1] = b   # index conversion
           
        elif j == n: 
#             print(i,j,n,2)
            g = row['susceptance']
            b = row['conductance']
            G[n-1, i-1] = g   # index conversion   
            B[n-1, i-1] = b   # index conversion

    return G, B

def build_Mp_Mq(G, B, n):
    Nb = G.shape[0]
    dim = 2 * Nb

    Mp = np.zeros((dim, dim))
    Mq = np.zeros((dim, dim))

    def add_sym(M, i, j, val):
        if i == j:
            M[i, j] += val
        else:
            M[i, j] += val / 2
            M[j, i] += val / 2

    nr = (n-1)     # index conversion
    ni = (n-1) + Nb  # index conversion

    for k in range(Nb):
        Gnk = G[n-1, k-1]  # index conversion
        Bnk = B[n-1, k-1]  # index conversion

        kr = k
        ki = k + Nb

        # ----- M^p_n -----
        add_sym(Mp, nr, kr,  Gnk)
        add_sym(Mp, nr, ki, -Bnk)
        add_sym(Mp, ni, ki,  Gnk)
        add_sym(Mp, ni, kr,  Bnk)

        # ----- M^q_n -----
        add_sym(Mq, ni, kr,  Gnk)
        add_sym(Mq, ni, ki, -Bnk)
        add_sym(Mq, nr, ki, -Gnk)
        add_sym(Mq, nr, kr, -Bnk)

    return Mp, Mq

def M_I_single(n, m, y_sq, num_buses):
    """
    Build M^I_nm matrix for a single branch (n, m).

    Parameters:
    - n, m: 0-based bus indices
    - y_sq: |Y_nm|^2 (scalar)
    - num_buses: total number of buses (Nb)

    Returns:
    - M: (2Nb x 2Nb) numpy array
    """
    dim = 2 * num_buses
    M = np.zeros((dim, dim))

    # Real part block
    M[n-1, n-1] += y_sq   # index conversion
    M[m-1, m-1] += y_sq   # index conversion
    M[n-1, m-1] -= y_sq   # index conversion
    M[m-1, n-1] -= y_sq   # index conversion

    # Imaginary part block
    n_i = num_buses + n
    m_i = num_buses + m

    M[n_i-1, n_i-1] += y_sq   # index conversion
    M[m_i-1, m_i-1] += y_sq   # index conversion
    M[n_i-1, m_i-1] -= y_sq   # index conversion
    M[m_i-1, n_i-1] -= y_sq   # index conversion

    return M

def M_v_single(n, num_buses):
    """
    M_v^n matrix.

    Parameters:
    - n: 0-based bus index
    - num_buses: Nb

    Returns:
    - M: (2Nb x 2Nb) numpy array
    """
    dim = 2 * num_buses
    M = np.zeros((dim, dim))

    M[n-1, n-1] = 1   # index conversion
    M[num_buses + n-1, num_buses + n-1] = 1   # index conversion

    return M

In [21]:
G_n, B_n = {}, {}
Mp_ng, Mq_ng = {}, {}

for n in case['gen']['bus_i']:
    G_n[n], B_n[n] = build_G_B_from_branch(n, case['branch'], nbus)
    Mp_ng[n], Mq_ng[n] = build_Mp_Mq(G_n[n], B_n[n], n)

G_n, B_n = {}, {}
Mp_nb, Mq_nb = {}, {}

for n in case['bus']['bus_i']:
    G_n[n], B_n[n] = build_G_B_from_branch(n, case['branch'], nbus)
    Mp_nb[n], Mq_nb[n] = build_Mp_Mq(G_n[n], B_n[n], n)

Mv_n = {}
for n in case['bus']['bus_i']:
    Mv_n[n]=M_v_single(n, nbus)

MI_nm={}
for line in case['branch'].index:
    n, m = case['branch'].loc[line,['bus_i','bus_j']]
    n, m = int(n), int(m)
    y_sq = case['branch'].loc[line,"y_sq"]
    MI_nm[(n, m)]=M_I_single(n, m, y_sq, nbus)

Ms = np.zeros((2*nbus,2*nbus))
Ms[nbus,nbus] = 1

Mc = np.zeros((2*nbus,2*nbus))
cp = dict(zip(case['gencost'].bus_i.values,case['gencost'].c1.values))
cq = dict(zip(case['gencost'].bus_i.values,case['gencost'].c1.values))

for n in case['gen'].bus_i:
    Mc += cp[n] * Mp_ng[n] + cq[n] * Mq_ng[n]
    
cost = case['gencost'].c1.values

pmax = case['gen'].Pmax.values / baseMVA
pmin = case['gen'].Pmin.values / baseMVA

qmax = case['gen'].Qmax.values / baseMVA
qmin = case['gen'].Qmin.values / baseMVA

Pd = case['bus'].Pd.values / baseMVA
Qd = case['bus'].Qd.values / baseMVA

Vmax = case['bus'].Vmax.values / baseMVA
Vmin = case['bus'].Vmin.values / baseMVA

Imax = case['branch'].I_max.values

Pd_gen, Qd_gen = [], []
for b in case['gen']['bus_i']:
    msk = case['bus']['bus_i']==b
    Pd_gen.append(case['bus'].loc[msk, 'Pd'].values)
    Qd_gen.append(case['bus'].loc[msk, 'Qd'].values)

# Data for Physics-Informed Neural Networks (PINNs)

In [22]:
import numpy as np
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
dtype = torch.float32

# ------------------------------------------------------------
# 1) Ordered bus sets
# ------------------------------------------------------------
all_bus_ids = [int(n) for n in case['bus']['bus_i']]

# unique generator buses only (important for bus-based constraints)
gen_bus_ids = sorted(set(int(n) for n in case['gen']['bus_i']))

# buses in N_b \ N_g
load_bus_ids = [b for b in all_bus_ids if b not in set(gen_bus_ids)]

# ordered branch keys matching your M_I construction
branch_ids = [
    (int(case['branch'].loc[line, 'bus_i']), int(case['branch'].loc[line, 'bus_j']))
    for line in case['branch'].index
]

# map bus ID -> position in full bus vector Pd/Qd
bus_id_to_pos = {bus_id: i for i, bus_id in enumerate(all_bus_ids)}

nbus = len(all_bus_ids)
ngen = len(gen_bus_ids)       # number of generator buses
nload = len(load_bus_ids)     # number of non-generator buses
nbranch = len(branch_ids)
d = 2 * nbus

# ------------------------------------------------------------
# 2) Stack quadratic matrices in the correct order
# ------------------------------------------------------------
M_p_gen_stack = torch.stack(
    [torch.as_tensor(Mp_ng[b], dtype=dtype, device=device) for b in gen_bus_ids],
    dim=0
)   # [ngen, d, d]

M_q_gen_stack = torch.stack(
    [torch.as_tensor(Mq_ng[b], dtype=dtype, device=device) for b in gen_bus_ids],
    dim=0
)   # [ngen, d, d]

M_p_load_stack = torch.stack(
    [torch.as_tensor(Mp_nb[b], dtype=dtype, device=device) for b in load_bus_ids],
    dim=0
)   # [nload, d, d]

M_q_load_stack = torch.stack(
    [torch.as_tensor(Mq_nb[b], dtype=dtype, device=device) for b in load_bus_ids],
    dim=0
)   # [nload, d, d]

M_v_stack = torch.stack(
    [torch.as_tensor(Mv_n[b], dtype=dtype, device=device) for b in all_bus_ids],
    dim=0
)   # [nbus, d, d]

M_I_stack = torch.stack(
    [torch.as_tensor(MI_nm[key], dtype=dtype, device=device) for key in branch_ids],
    dim=0
)   # [nbranch, d, d]

M_c = torch.as_tensor(Mc, dtype=dtype, device=device)      # [d, d]
M_s = torch.as_tensor(Ms, dtype=dtype, device=device)      # [d, d]

# ------------------------------------------------------------
# 3) Symmetrize matrices (recommended)
# ------------------------------------------------------------
M_c = 0.5 * (M_c + M_c.T)
M_p_gen_stack = 0.5 * (M_p_gen_stack + M_p_gen_stack.transpose(-1, -2))
M_q_gen_stack = 0.5 * (M_q_gen_stack + M_q_gen_stack.transpose(-1, -2))
M_p_load_stack = 0.5 * (M_p_load_stack + M_p_load_stack.transpose(-1, -2))
M_q_load_stack = 0.5 * (M_q_load_stack + M_q_load_stack.transpose(-1, -2))
M_v_stack = 0.5 * (M_v_stack + M_v_stack.transpose(-1, -2))
M_I_stack = 0.5 * (M_I_stack + M_I_stack.transpose(-1, -2))
# M_s usually does not need symmetrization here, but harmless if desired:
# M_s = 0.5 * (M_s + M_s.T)

# ------------------------------------------------------------
# 4) Bus-level demand vectors
# ------------------------------------------------------------
Pd_bus = np.asarray(case['bus'].Pd.values, dtype=np.float32) / baseMVA
Qd_bus = np.asarray(case['bus'].Qd.values, dtype=np.float32) / baseMVA

# extract in the same order as the matrix stacks
Pd_load = np.asarray([Pd_bus[bus_id_to_pos[b]] for b in load_bus_ids], dtype=np.float32)
Qd_load = np.asarray([Qd_bus[bus_id_to_pos[b]] for b in load_bus_ids], dtype=np.float32)

Pd_gen = np.asarray([Pd_bus[bus_id_to_pos[b]] for b in gen_bus_ids], dtype=np.float32)
Qd_gen = np.asarray([Qd_bus[bus_id_to_pos[b]] for b in gen_bus_ids], dtype=np.float32)

# ------------------------------------------------------------
# 5) Generator bounds aggregated per generator bus
#    (important if multiple generators are connected to one bus)
# ------------------------------------------------------------
pmax_by_bus, pmin_by_bus = {}, {}
qmax_by_bus, qmin_by_bus = {}, {}

for _, row in case['gen'].iterrows():
    b = int(row['bus_i'])
    pmax_by_bus[b] = pmax_by_bus.get(b, 0.0) + float(row['Pmax']) / baseMVA
    pmin_by_bus[b] = pmin_by_bus.get(b, 0.0) + float(row['Pmin']) / baseMVA
    qmax_by_bus[b] = qmax_by_bus.get(b, 0.0) + float(row['Qmax']) / baseMVA
    qmin_by_bus[b] = qmin_by_bus.get(b, 0.0) + float(row['Qmin']) / baseMVA

pmax_g = np.asarray([pmax_by_bus[b] for b in gen_bus_ids], dtype=np.float32)
pmin_g = np.asarray([pmin_by_bus[b] for b in gen_bus_ids], dtype=np.float32)
qmax_g = np.asarray([qmax_by_bus[b] for b in gen_bus_ids], dtype=np.float32)
qmin_g = np.asarray([qmin_by_bus[b] for b in gen_bus_ids], dtype=np.float32)

# ------------------------------------------------------------
# 6) Voltage / branch limits
# ------------------------------------------------------------
# keep exactly your scaling convention
Vmax_arr = np.asarray(case['bus'].Vmax.values, dtype=np.float32) / baseMVA
Vmin_arr = np.asarray(case['bus'].Vmin.values, dtype=np.float32) / baseMVA
Imax_arr = np.asarray(case['branch'].I_max.values, dtype=np.float32)

# ------------------------------------------------------------
# 7) Tensor index positions for extracting batched Pd/Qd inside loss
# ------------------------------------------------------------
load_bus_pos = torch.tensor(
    [bus_id_to_pos[b] for b in load_bus_ids],
    dtype=torch.long,
    device=device
)

gen_bus_pos = torch.tensor(
    [bus_id_to_pos[b] for b in gen_bus_ids],
    dtype=torch.long,
    device=device
)

# ------------------------------------------------------------
# 8) kappa
# ------------------------------------------------------------
kappa_value = 0.0
if 'kappa' in locals():
    kappa_value = float(kappa)

# ------------------------------------------------------------
# 9) Final problem dictionary for the PINN loss
# ------------------------------------------------------------
problem = {
    "M_c": M_c,
    "M_p_load": M_p_load_stack,
    "M_q_load": M_q_load_stack,
    "M_p_gen": M_p_gen_stack,
    "M_q_gen": M_q_gen_stack,
    "M_v": M_v_stack,
    "M_I": M_I_stack,
    "M_s": M_s,

    # base vectors (useful for reference/debugging)
    "Pd_bus": torch.as_tensor(Pd_bus, dtype=dtype, device=device),
    "Qd_bus": torch.as_tensor(Qd_bus, dtype=dtype, device=device),
    "Pd_load": torch.as_tensor(Pd_load, dtype=dtype, device=device),
    "Qd_load": torch.as_tensor(Qd_load, dtype=dtype, device=device),
    "Pd_gen": torch.as_tensor(Pd_gen, dtype=dtype, device=device),
    "Qd_gen": torch.as_tensor(Qd_gen, dtype=dtype, device=device),

    "pmax_g": torch.as_tensor(pmax_g, dtype=dtype, device=device),
    "pmin_g": torch.as_tensor(pmin_g, dtype=dtype, device=device),
    "qmax_g": torch.as_tensor(qmax_g, dtype=dtype, device=device),
    "qmin_g": torch.as_tensor(qmin_g, dtype=dtype, device=device),

    "Vmax": torch.as_tensor(Vmax_arr, dtype=dtype, device=device),
    "Vmin": torch.as_tensor(Vmin_arr, dtype=dtype, device=device),
    "Imax": torch.as_tensor(Imax_arr, dtype=dtype, device=device),

    # use these inside compute_loss_aug_lagrangian:
    # Pd_load_batch = Pd_batch[:, problem["load_bus_pos"]]
    # Pd_gen_batch  = Pd_batch[:, problem["gen_bus_pos"]]
    "load_bus_pos": load_bus_pos,
    "gen_bus_pos": gen_bus_pos,

    "kappa": torch.as_tensor(kappa_value, dtype=dtype, device=device),

    # metadata
    "all_bus_ids": all_bus_ids,
    "gen_bus_ids": gen_bus_ids,
    "load_bus_ids": load_bus_ids,
    "branch_ids": branch_ids,
}

print("Constructed PINN problem data:")
print(f"  nbus   = {nbus}")
print(f"  ngen   = {ngen}   (generator buses)")
print(f"  nload  = {nload}  (non-generator buses)")
print(f"  nbranch= {nbranch}")
print(f"  M_p_gen shape  = {tuple(problem['M_p_gen'].shape)}")
print(f"  M_p_load shape = {tuple(problem['M_p_load'].shape)}")
print(f"  M_v shape      = {tuple(problem['M_v'].shape)}")
print(f"  M_I shape      = {tuple(problem['M_I'].shape)}")
print(f"  load_bus_pos   = {problem['load_bus_pos'].tolist()}")
print(f"  gen_bus_pos    = {problem['gen_bus_pos'].tolist()}")

Constructed PINN problem data:
  nbus   = 3
  ngen   = 2   (generator buses)
  nload  = 1  (non-generator buses)
  nbranch= 3
  M_p_gen shape  = (2, 6, 6)
  M_p_load shape = (1, 6, 6)
  M_v shape      = (3, 6, 6)
  M_I shape      = (3, 6, 6)
  load_bus_pos   = [2]
  gen_bus_pos    = [0, 1]


# Random Demands for PINN

In [23]:
import torch

def gaussian_batch(base_tensor, batch_size, variation_std=0.05, clamp_min=0.0):
    """
    Create a batch of tensors with Gaussian random variations.
    
    Args:
        base_tensor (torch.Tensor): original tensor of shape [N]
        batch_size (int): number of samples in batch
        variation_std (float): standard deviation of Gaussian noise, relative to base_tensor
        clamp_min (float): minimum allowed value for perturbed tensor
    
    Returns:
        torch.Tensor: batch of shape [batch_size, N]
    """
    # Expand base tensor to batch
    base_batch = base_tensor.unsqueeze(0).repeat(batch_size, 1)
    
    # Gaussian noise scaled by base_tensor
    noise = variation_std * base_tensor.unsqueeze(0) * torch.randn_like(base_batch)
    
    # Perturbed batch
    batch = base_batch + noise
    
    # Clamp if necessary
    if clamp_min is not None:
        batch = torch.clamp(batch, min=clamp_min)
    
    return batch


def uniform_batch(base_tensor, batch_size, variation=0.05, clamp_min=0.0):
    """
    Create a batch of tensors with uniform random variations.
    
    Args:
        base_tensor (torch.Tensor): original tensor of shape [N]
        batch_size (int): number of samples in batch
        variation (float): maximum relative deviation (±variation)
        clamp_min (float): minimum allowed value for perturbed tensor
    
    Returns:
        torch.Tensor: batch of shape [batch_size, N]
    """
    base_batch = base_tensor.unsqueeze(0).repeat(batch_size, 1)
    
    # Uniform noise in [-variation, +variation]
    noise = (2 * torch.rand_like(base_batch) - 1) * variation
    
    batch = base_batch * (1 + noise)
    
    if clamp_min is not None:
        batch = torch.clamp(batch, min=clamp_min)
    
    return batch

# Baseline NN framework - physics for feasibility only

In [24]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim


# ------------------------------------------------------------
# Helper functions
# ------------------------------------------------------------

def quad_batch(v: torch.Tensor, M: torch.Tensor) -> torch.Tensor:
    # v: [B, d], M: [d, d] -> [B]
    return torch.einsum("bi,ij,bj->b", v, M, v)

def quad_batch_stack(v: torch.Tensor, M: torch.Tensor) -> torch.Tensor:
    # v: [B, d], M: [K, d, d] -> [B, K]
    return torch.einsum("bi,kij,bj->bk", v, M, v)

def gaussian_batch(x: torch.Tensor, batch_size: int, variation_std: float = 0.05) -> torch.Tensor:
    # x: [n] -> [B, n]
    x = x.reshape(1, -1).expand(batch_size, -1)
    scale = torch.clamp(x.abs(), min=1.0)
    noise = variation_std * torch.randn_like(x) * scale
    return x + noise

def to_row_batch(x: torch.Tensor, B: int) -> torch.Tensor:
    # robust expansion for vectors that may be [N], [N,1], [1,N], etc.
    return x.reshape(1, -1).expand(B, -1)


# ------------------------------------------------------------
# Verification-oriented feasibility network
# ------------------------------------------------------------

class FeasibleVoltageMLP(nn.Module):
    """
    Input:
        Pd: [B, nbus]
        Qd: [B, nbus]

    Output:
        v:  [B, 2*nbus]

    Architecture choices for verification-readiness:
      - only affine + ReLU + clamp
      - no dual variables
      - no implicit optimization layer
      - no eig/SVD/nullspace operations
      - slack imaginary part fixed to zero by construction
    """
    def __init__(
        self,
        nbus: int,
        hidden1: int = 256,
        hidden2: int = 256,
        hidden3: int = 256,
        slack_imag_idx: int = None,
        bound_margin: float = 0.98,   # keep outputs slightly inside box
    ):
        super().__init__()
        self.nbus = nbus
        self.in_dim = 2 * nbus
        self.out_dim = 2 * nbus
        self.slack_imag_idx = nbus if slack_imag_idx is None else int(slack_imag_idx)
        self.bound_margin = float(bound_margin)

        self.net = nn.Sequential(
            nn.Linear(self.in_dim, hidden1),
            nn.ReLU(),
            nn.Linear(hidden1, hidden2),
            nn.ReLU(),
            nn.Linear(hidden2, hidden3),
            nn.ReLU(),
            nn.Linear(hidden3, self.out_dim),
        )

    def forward(self, Pd: torch.Tensor, Qd: torch.Tensor, Vmax: torch.Tensor) -> torch.Tensor:
        """
        Vmax: [nbus] or [B, nbus]
        Returns v = [vr, vi] with each component bounded in [-bound_margin*Vmax, +bound_margin*Vmax]
        and slack imaginary component exactly zero.
        """
        B = Pd.shape[0]
        x = torch.cat([Pd, Qd], dim=-1)            # [B, 2*nbus]
        raw = self.net(x)                          # [B, 2*nbus]

        vr_raw = raw[:, :self.nbus]                # [B, nbus]
        vi_raw = raw[:, self.nbus:]                # [B, nbus]

        Vmax_b = Vmax.reshape(1, -1).expand(B, -1) if Vmax.dim() == 1 else Vmax
        comp_bound = self.bound_margin * Vmax_b

        # Piecewise-linear bounded map: clamp is simpler than tanh/sigmoid for verification-oriented use
        vr = torch.clamp(vr_raw, min=-comp_bound, max=comp_bound)
        vi = torch.clamp(vi_raw, min=-comp_bound, max=comp_bound)

        # Enforce slack imaginary part = 0 exactly
        vi = vi.clone()
        vi[:, self.slack_imag_idx - self.nbus if self.slack_imag_idx >= self.nbus else 0] = 0.0
        # In your previous Ms, Ms[nbus, nbus] = 1, so slack_imag_idx = nbus means first imag entry.
        # The expression above converts the global index into the local imag-block index.

        v = torch.cat([vr, vi], dim=-1)            # [B, 2*nbus]
        return v


# ------------------------------------------------------------
# Pure feasibility loss
# ------------------------------------------------------------

def compute_feasibility_loss(
    model: nn.Module,
    Pd_batch: torch.Tensor,
    Qd_batch: torch.Tensor,
    problem: dict,
    w_eq_p: float = 10.0,
    w_eq_q: float = 10.0,
    w_ineq_gen: float = 5.0,
    w_ineq_v: float = 5.0,
    w_ineq_I: float = 5.0,
    w_slack: float = 20.0,
    w_obj: float = 0.01,
):
    B = Pd_batch.shape[0]
    device = Pd_batch.device

    # --------------------------------------------------------
    # pull problem data
    # --------------------------------------------------------
    M_c      = problem["M_c"]          # [d,d]
    M_p_load = problem["M_p_load"]     # [nload,d,d]
    M_q_load = problem["M_q_load"]     # [nload,d,d]
    M_p_gen  = problem["M_p_gen"]      # [ngen,d,d]
    M_q_gen  = problem["M_q_gen"]      # [ngen,d,d]
    M_v      = problem["M_v"]          # [nbus,d,d]
    M_I      = problem["M_I"]          # [nbranch,d,d]
    M_s      = problem["M_s"]          # [d,d]

    load_bus_pos = problem["load_bus_pos"]   # [nload]
    gen_bus_pos  = problem["gen_bus_pos"]    # [ngen]

    pmax_g = to_row_batch(problem["pmax_g"], B)   # [B,ngen]
    pmin_g = to_row_batch(problem["pmin_g"], B)
    qmax_g = to_row_batch(problem["qmax_g"], B)
    qmin_g = to_row_batch(problem["qmin_g"], B)
    Vmax   = to_row_batch(problem["Vmax"], B)     # [B,nbus]
    Vmin   = to_row_batch(problem["Vmin"], B)
    Imax   = to_row_batch(problem["Imax"], B)     # [B,nbranch]

    # --------------------------------------------------------
    # sample-dependent demand values
    # --------------------------------------------------------
    Pd_load = Pd_batch[:, load_bus_pos]      # [B,nload]
    Qd_load = Qd_batch[:, load_bus_pos]      # [B,nload]
    Pd_gen  = Pd_batch[:, gen_bus_pos]       # [B,ngen]
    Qd_gen  = Qd_batch[:, gen_bus_pos]       # [B,ngen]

    # --------------------------------------------------------
    # predict primal voltage
    # --------------------------------------------------------
    v = model(Pd_batch, Qd_batch, Vmax=problem["Vmax"].reshape(-1))

    # --------------------------------------------------------
    # constraint values
    # --------------------------------------------------------
    vp_load = quad_batch_stack(v, M_p_load)   # [B,nload]
    vq_load = quad_batch_stack(v, M_q_load)   # [B,nload]
    vp_gen  = quad_batch_stack(v, M_p_gen)    # [B,ngen]
    vq_gen  = quad_batch_stack(v, M_q_gen)    # [B,ngen]
    vv      = quad_batch_stack(v, M_v)        # [B,nbus]
    vI      = quad_batch_stack(v, M_I)        # [B,nbranch]
    vs      = quad_batch(v, M_s).unsqueeze(-1)

    # Equalities on non-generator buses
    h_pb = vp_load + Pd_load
    h_qb = vq_load + Qd_load

    # Generator inequalities
    g_pg_u = vp_gen - (pmax_g - Pd_gen)
    g_pg_l = -vp_gen + (pmin_g - Pd_gen)
    g_qg_u = vq_gen - (qmax_g - Qd_gen)
    g_qg_l = -vq_gen + (qmin_g - Qd_gen)

    # Voltage / current inequalities
    g_v_u = vv - Vmax.pow(2)
    g_v_l = -vv + Vmin.pow(2)
    g_I   = vI - Imax.pow(2)

    # Slack equality
    h_s = vs

    # --------------------------------------------------------
    # residual losses
    # --------------------------------------------------------
    loss_eq_p = h_pb.pow(2).mean()
    loss_eq_q = h_qb.pow(2).mean()

    loss_gen_ineq = (
        F.relu(g_pg_u).pow(2).mean()
        + F.relu(g_pg_l).pow(2).mean()
        + F.relu(g_qg_u).pow(2).mean()
        + F.relu(g_qg_l).pow(2).mean()
    )

    loss_v_ineq = (
        F.relu(g_v_u).pow(2).mean()
        + F.relu(g_v_l).pow(2).mean()
    )

    loss_I_ineq = F.relu(g_I).pow(2).mean()

    # h_s should already be zero by construction if Ms selects the slack imaginary component,
    # but keep this term as a safety check.
    loss_slack = h_s.pow(2).mean()

    # small objective regularizer (optional, to bias toward cheaper feasible points)
    obj = quad_batch(v, M_c).mean()

    total_loss = (
        w_eq_p * loss_eq_p
        + w_eq_q * loss_eq_q
        + w_ineq_gen * loss_gen_ineq
        + w_ineq_v * loss_v_ineq
        + w_ineq_I * loss_I_ineq
        + w_slack * loss_slack
        + w_obj * obj
    )

    diagnostics = {
        "loss_total": total_loss.detach(),
        "loss_eq_p": loss_eq_p.detach(),
        "loss_eq_q": loss_eq_q.detach(),
        "loss_gen_ineq": loss_gen_ineq.detach(),
        "loss_v_ineq": loss_v_ineq.detach(),
        "loss_I_ineq": loss_I_ineq.detach(),
        "loss_slack": loss_slack.detach(),
        "obj": obj.detach(),
        "max_abs_h_pb": h_pb.abs().max().detach(),
        "max_abs_h_qb": h_qb.abs().max().detach(),
        "max_g_pg_u": g_pg_u.max().detach(),
        "max_g_pg_l": g_pg_l.max().detach(),
        "max_g_qg_u": g_qg_u.max().detach(),
        "max_g_qg_l": g_qg_l.max().detach(),
        "max_g_v_u": g_v_u.max().detach(),
        "max_g_v_l": g_v_l.max().detach(),
        "max_g_I": g_I.max().detach(),
    }

    return total_loss, diagnostics


# ------------------------------------------------------------
# Build model
# ------------------------------------------------------------

device = problem["M_c"].device
nbus = len(problem["all_bus_ids"])

# In your previous construction Ms[nbus, nbus] = 1, so global slack imag index is nbus
slack_imag_idx = nbus

model = FeasibleVoltageMLP(
    nbus=nbus,
    hidden1=10*nbus,
    hidden2=5*nbus,
    hidden3=10*nbus,
    slack_imag_idx=slack_imag_idx,
    bound_margin=0.98,
).to(device)

optimizer = optim.Adam(model.parameters(), lr=1e-3)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=1000, gamma=0.5)

Pd_bus_base = problem["Pd_bus"].reshape(-1)
Qd_bus_base = problem["Qd_bus"].reshape(-1)




## Naive Training

In [25]:
# ------------------------------------------------------------
# Training loop
# ------------------------------------------------------------

num_epochs = 3000
batch_size = 32
best_loss = float("inf")
best_state = None

for epoch in range(num_epochs):
    # sample demand scenarios
    Pd_batch = gaussian_batch(Pd_bus_base, batch_size=batch_size, variation_std=0.05)
    Qd_batch = gaussian_batch(Qd_bus_base, batch_size=batch_size, variation_std=0.05)

    # demands should remain nonnegative
    Pd_batch = torch.clamp(Pd_batch, min=0.0)
    Qd_batch = torch.clamp(Qd_batch, min=0.0)

    optimizer.zero_grad()

    loss, diagnostics = compute_feasibility_loss(
        model=model,
        Pd_batch=Pd_batch,
        Qd_batch=Qd_batch,
        problem=problem,
        w_eq_p=10.0,
        w_eq_q=10.0,
        w_ineq_gen=5.0,
        w_ineq_v=5.0,
        w_ineq_I=5.0,
        w_slack=20.0,
        w_obj=0.01,
    )

    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=10.0)
    optimizer.step()
    scheduler.step()

    loss_value = loss.detach().item()
    if loss_value < best_loss:
        best_loss = loss_value
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

    if epoch % 100 == 0:
        print(f"\nEpoch {epoch:5d} | loss = {loss_value:.6e}")
        print({k: v.detach().item() for k, v in diagnostics.items()})

# restore best model
if best_state is not None:
    model.load_state_dict(best_state)

print("\nBest training loss:", best_loss)





Epoch     0 | loss = 1.164052e+01
{'loss_total': 11.640517234802246, 'loss_eq_p': 0.9140639901161194, 'loss_eq_q': 0.2499864548444748, 'loss_gen_ineq': 0.0, 'loss_v_ineq': 8.275804042057189e-09, 'loss_I_ineq': 0.0, 'loss_slack': 0.0, 'obj': 0.001197085715830326, 'max_abs_h_pb': 1.1069296598434448, 'max_abs_h_qb': 0.5924140214920044, 'max_g_pg_u': -18.698993682861328, 'max_g_pg_l': -0.9919765591621399, 'max_g_qg_u': -9.467384338378906, 'max_g_qg_l': -10.288619041442871, 'max_g_v_u': 0.00011141680442960933, 'max_g_v_l': -3.52084098267369e-05, 'max_g_I': -14467.5908203125}

Epoch   100 | loss = 1.144617e+01
{'loss_total': 11.446171760559082, 'loss_eq_p': 0.8885504007339478, 'loss_eq_q': 0.25606560707092285, 'loss_gen_ineq': 0.0, 'loss_v_ineq': 8.275804042057189e-09, 'loss_I_ineq': 0.0, 'loss_slack': 0.0, 'obj': 0.001197085715830326, 'max_abs_h_pb': 1.024787425994873, 'max_abs_h_qb': 0.5934157371520996, 'max_g_pg_u': -18.76945686340332, 'max_g_pg_l': -0.9597128629684448, 'max_g_qg_u': -9.

## Final evaluation

In [26]:
# ------------------------------------------------------------
# Final evaluation on one fresh batch
# ------------------------------------------------------------

with torch.no_grad():
    Pd_test = gaussian_batch(Pd_bus_base, batch_size=64, variation_std=0.05)
    Qd_test = gaussian_batch(Qd_bus_base, batch_size=64, variation_std=0.05)
    Pd_test = torch.clamp(Pd_test, min=0.0)
    Qd_test = torch.clamp(Qd_test, min=0.0)

    test_loss, test_diag = compute_feasibility_loss(
        model=model,
        Pd_batch=Pd_test,
        Qd_batch=Qd_test,
        problem=problem,
        w_eq_p=10.0,
        w_eq_q=10.0,
        w_ineq_gen=5.0,
        w_ineq_v=5.0,
        w_ineq_I=5.0,
        w_slack=20.0,
        w_obj=0.01,
    )

print("\nFinal test diagnostics:")
print({k: v.detach().item() for k, v in test_diag.items()})

# The trained object you want for later verification is `model`.
# A verification pipeline can then check, over an input demand set,
# whether the resulting v satisfies:
#   h_pb = 0, h_qb = 0,
#   g_pg_u <= 0, g_pg_l <= 0,
#   g_qg_u <= 0, g_qg_l <= 0,
#   g_v_u <= 0, g_v_l <= 0,
#   g_I <= 0,
#   h_s = 0.


Final test diagnostics:
{'loss_total': 11.528228759765625, 'loss_eq_p': 0.902121365070343, 'loss_eq_q': 0.25070029497146606, 'loss_gen_ineq': 0.0, 'loss_v_ineq': 8.275804042057189e-09, 'loss_I_ineq': 0.0, 'loss_slack': 0.0, 'obj': 0.001197085715830326, 'max_abs_h_pb': 1.064841866493225, 'max_abs_h_qb': 0.6144532561302185, 'max_g_pg_u': -18.761356353759766, 'max_g_pg_l': -0.9387808442115784, 'max_g_qg_u': -9.453756332397461, 'max_g_qg_l': -10.188384056091309, 'max_g_v_u': 0.00011141680442960933, 'max_g_v_l': -3.52084098267369e-05, 'max_g_I': -14467.5908203125}


# EON